In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Lokaler Testmodus von Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>

Verwende den lokalen Testmodus (verfügbar ab `qiskit-ibm-runtime` v0.22.0) um Programme zu testen, bevor du sie feinjustierst und an echte Quantenhardware sendest. Nachdem du deinen Code im lokalen Testmodus verifiziert hast, musst du lediglich den Backend-Namen ändern, um ihn auf einem QPU auszuführen.

Um den lokalen Testmodus zu nutzen, gib eines der Fake-Backends aus ``qiskit_ibm_runtime.fake_provider`` an oder verwende ein Qiskit Aer Backend, wenn du ein Qiskit Runtime Primitive oder eine Session instantiierst.

- **Fake Backends**: Die [Fake Backends](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) in ``qiskit_ibm_runtime.fake_provider`` ahmen das Verhalten von IBM&reg; QPUs mithilfe von QPU-Snapshots nach. Die QPU-Snapshots enthalten wichtige Informationen über den QPU, wie z. B. die Coupling Map, Basis-Gates und Qubit-Eigenschaften – nützlich zum Testen des Transpilers und für verrauschte Simulationen des QPU. Das Rauschmodell aus dem Snapshot wird während der Simulation automatisch angewendet.
- **Aer-Simulator**: Simulatoren aus [Qiskit Aer](/guides/simulate-with-qiskit-aer) bieten leistungsstärkere Simulationen, die größere Circuits und [benutzerdefinierte Rauschmodelle](/guides/build-noise-models) verarbeiten können. Im lokalen Testmodus stehen dir verschiedene Simulationsmethoden zur Verfügung, wenn du `AerSimulator` verwendest. Sieh dir das [Clifford-Simulationsmodus-Beispiel](#clifford-sim) an, das zeigt, wie Clifford-Circuits mit einer großen Anzahl an Qubits effizient simuliert werden können.

    <details>
    <summary>**Liste der verfügbaren Simulationsmethoden von Qiskit Aer**</summary>

    Weitere Informationen findest du in der [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator)-Dokumentation.

    * ``"automatic"``: Standard-Simulationsmethode. Wählt die Simulationsmethode automatisch basierend auf Circuit und Rauschmodell aus.

    * ``"statevector"``: Eine dichte Zustandsvektor-Simulation, die Messergebnisse aus *idealen* Circuits mit allen Messungen am Ende des Circuits sampeln kann. Bei verrauschten Simulationen wird für jeden Shot ein zufällig ausgewählter verrauschter Circuit aus dem Rauschmodell gesampelt.

    * ``"density_matrix"``: Eine Dichtematrix-Simulation, die Messergebnisse aus *verrauschten* Circuits mit allen Messungen am Ende des Circuits sampeln kann.

    * ``"stabilizer"``: Ein effizienter Clifford-Stabilizer-Zustandssimulator, der verrauschte Clifford-Circuits simulieren kann, wenn alle Fehler im Rauschmodell ebenfalls Clifford-Fehler sind.

    * ``"extended_stabilizer"``: Ein approximativer Simulator für Clifford-+T-Circuits, der auf der Zerlegung des Zustands in einen Rang-Stabilizer-Zustand basiert. Die Anzahl der Terme wächst mit der Anzahl der Nicht-Clifford-Gates (T).

    * ``"matrix_product_state"``: Ein Tensor-Netzwerk-Zustandsvektorsimulator, der eine Matrix Product State (MPS)-Darstellung für den Zustand verwendet. Dies kann mit oder ohne Abschneiden der MPS-Binddimensionen erfolgen, abhängig von den Simulatoroptionen. Standardmäßig wird nicht abgeschnitten.

    * ``"unitary"``: Eine dichte unitäre Matrixsimulation eines idealen Circuits. Diese simuliert die unitäre Matrix des Circuits selbst und nicht die Entwicklung eines Anfangszustands. Diese Methode kann nur Gates simulieren; sie unterstützt keine Messungen, Resets oder Rauschen.

    * ``"superop"``: Eine dichte Superoperator-Matrixsimulation eines idealen oder verrauschten Circuits. Diese simuliert die Superoperatormatrix des Circuits selbst und nicht die Entwicklung eines Anfangszustands. Diese Methode kann ideale und verrauschte Gates sowie Resets simulieren, unterstützt jedoch keine Messungen.

    * ``"tensor_network"``: Eine auf Tensor-Netzwerken basierende Simulation, die sowohl Zustandsvektor als auch Dichtematrix unterstützt. Derzeit ist diese nur für GPU verfügbar und wird durch die Verwendung von cuQuantum `cuTensorNet` APIs beschleunigt.
    </details>

> **Note:** - Du kannst alle Qiskit Runtime-Optionen im lokalen Testmodus angeben. Allerdings werden alle Optionen außer shots ignoriert, wenn auf einem lokalen Simulator ausgeführt wird.
> - Es wird empfohlen, Qiskit Aer vor der Verwendung von Fake Backends oder Aer-Simulatoren zu installieren, indem du `pip install qiskit-aer` ausführst. Die Fake Backends verwenden intern Aer-Simulatoren, sofern verfügbar, um deren Leistungsvorteile zu nutzen.


## Fake-Backends-Beispiel

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## AerSimulator-Beispiele
Beispiel mit Sessions, ohne Rauschen:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Um mit Rauschen zu simulieren, gib einen QPU (Quantenhardware) an und übergib ihn an Aer. Aer erstellt dann ein Rauschmodell basierend auf den Kalibrierungsdaten dieses QPU und instantiiert ein Aer-Backend mit diesem Modell. Falls du es vorziehst, kannst du auch ein [Rauschmodell selbst erstellen](/guides/build-noise-models).

> **Caution:** Ein QPU kann von verschiedenen Arten von Rauschen betroffen sein. Das hier verwendete Qiskit Aer-Rauschmodell simuliert nur einen Teil davon und ist daher wahrscheinlich weniger stark ausgeprägt als das Rauschen auf einem echten QPU.
> 
> Einzelheiten dazu, welche Fehler beim Initialisieren eines Rauschmodells aus einem QPU berücksichtigt werden, findest du in der Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend) API-Referenz.

Beispiel mit Rauschen:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Clifford-Simulation
Da Clifford-Circuits effizient mit verifizierbaren Ergebnissen simuliert werden können, ist die Clifford-Simulation ein sehr nützliches Werkzeug. Ein ausführlicheres Beispiel findest du unter [Effiziente Simulation von Stabilizer-Circuits mit Qiskit Aer Primitives](/guides/simulate-stabilizer-circuits).

Beispiel:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Nächste Schritte
> **Tip:** - Sieh dir detaillierte [Primitives-Beispiele](/guides/primitives-examples) an.
>     - Lies [Zu V2 Primitives migrieren](/guides/v2-primitives).
>     - Übe mit Primitives, indem du die [Kostenfunktions-Lektion](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning durcharbeitest.
>     - Lerne, wie du lokal transpilierst, im Abschnitt [Transpile](/guides/transpile).
>     - Probiere das Tutorial [Transpiler-Einstellungen vergleichen](/guides/circuit-transpilation-settings#compare-transpiler-settings) aus.